In [1]:
#!python 3.9.7

%matplotlib qt

import logging

log_level = logging.DEBUG # If you find anything is not going right, try logging.DEBUG. Otherwise INFO is the default.
NUM_POINTS_PER_CONTOUR = 15 # Numbers of points per contour. Is only the starting value.

# Standard library imports
import copy
import sys
import threading
import time
from os import listdir, _exit
from os.path import isfile, join
from pprint import pprint
from threading import Thread
from tkinter import filedialog, messagebox

# Third party imports
import cv2
import matplotlib.pyplot as plt
from matplotlib.widgets import (
    Button,
    PolygonSelector,
    RectangleSelector,
    Slider,
    TextBox
)
import numpy as np
import pandas as pd
from PIL import Image
from PIL.TiffTags import TAGS
from scipy.interpolate import splprep, splev
from scipy.ndimage import fourier_shift
from skimage import (
    filters,
    io,
    measure, 
    morphology
)
from skimage.registration import phase_cross_correlation
import tkinter as tk


In [2]:
def get_custom_logger(name: str = "FRAP_Logger", level=logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)

    # Prevent propagation to root logger (avoids messages from other libraries like matplotlib)
    logger.propagate = False

    # Only add handler if not already present (avoids duplicate logs)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stderr)
        formatter = logging.Formatter("[%(levelname)s] %(name)s: %(message)s")
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

logger = get_custom_logger(level=log_level)

In [3]:
MANUALMODE = False

if MANUALMODE:
    root = tk.Tk()
    root.wm_attributes('-topmost', True)
    root.withdraw()
    manualpath = filedialog.askopenfilename(filetypes=[('Tiff-Files','*.tif'),('ALL files','*')])


In [4]:
if MANUALMODE: 
    image = io.imread(manualpath)
    plt.subplots(figsize=(6,6))
    plt.imshow(image[0,:,:])
    plt.show()


In [5]:
"""
Functions for FRAP (Fluorescence Recovery After Photobleaching) analysis.

This module provides functions to:
1. Detect FRAP bleaching frames in an image sequence
2. Calculate and refine regions of interest (ROIs) 
3. Export intensity measurements for FRAP analysis

Authors: damlatetiker, johannbrenner, jeremypflaum
Created: Nov 1, 2022
"""

def detect_frap(curr_img):
    """
    Determine the last frame before and first frame after photobleaching.
    
    The function works by:
    1. Creating a rough estimate of the bleached condensate via thresholding
    2. Summing pixel intensities in this ROI over the whole time series
    3. Finding the sudden intensity drop that indicates bleaching
    
    Args:
        curr_img: 3D numpy array (time, height, width) containing the image sequence
        
    Returns:
        tuple: (ind_before, ind_after)
            ind_before: Index of last frame before bleaching
            ind_after: Index of first frame after bleaching
    """
    # Get first frame and determine threshold using Li's method
    first_frame = curr_img[0].astype(float)
    thresh_li = filters.threshold_li(first_frame)
    
    # Create binary mask and remove small objects
    mask = first_frame > thresh_li
    mask = morphology.remove_small_objects(mask, 15)
    
    # Calculate intensity time series within masked region
    int_seq = []
    for i in range(len(curr_img)):
        int_seq.append(np.sum(np.where(mask==True, curr_img[i], 0)))
    int_seq = np.array(int_seq).astype(float)
    
    # Find bleaching point from intensity derivative
    der = np.diff(int_seq)
    ind_before = np.argmin(der)
    ind_after = ind_before + 1

    if log_level < 20:
        logger.debug("Index before bleach: {:d}".format(ind_before))
        logger.debug("Index after bleach: {:d}".format(ind_after))
        
    return ind_before, ind_after


def refine_ROIs(curr_img, ind_before, ind_after):
    """
    Refine the regions of interest for FRAP analysis.
    
    Calculates two ROIs:
    1. Total cell/condensate area
    2. Bleached region
    
    Args:
        curr_img: 3D numpy array of images
        ind_before: Index of last frame before bleaching
        ind_after: Index of first frame after bleaching
        
    Returns:
        tuple: (mask_roi1, mask_roi2)
            mask_roi1: Binary mask of total cell area
            mask_roi2: Binary mask of bleached region
    """
    
    # Step 1: Average pre-bleach frames
    if ind_before == 0:
        frames_before = curr_img[ind_before].astype(float)
        img_avg = frames_before
    else:
        frames_before = curr_img[:ind_before].astype(float)
        img_avg = np.mean(frames_before, axis=0)
    thresh_li1 = filters.threshold_li(img_avg) 
    mask_roi1 = img_avg > thresh_li1
    mask_roi1 = morphology.remove_small_objects(mask_roi1, 10)


    if log_level < 20:
        logger.debug(f"Step 1: Pre-bleach average threshold = {thresh_li1:.3f}")
        plt.figure()
        plt.title("Step 1: Pre-bleach avg image")
        plt.imshow(img_avg, cmap='gray')
        plt.colorbar()
        plt.figure()
        plt.title("Step 1: ROI1 (total area mask)")
        plt.imshow(mask_roi1, cmap='gray')
        plt.show()

    # Step 2: Threshold first post-bleach frame
    first_after = curr_img[ind_after].astype(float)
    thresh_li2 = filters.threshold_li(first_after)
    mask_int = first_after > thresh_li2
    mask_int = morphology.remove_small_objects(mask_int, 10)


    if log_level < 20:
        logger.debug(f"Step 2: Post-bleach threshold = {thresh_li2:.3f}")
        plt.figure()
        plt.title("Step 2: First post-bleach image")
        plt.imshow(first_after, cmap='gray')
        plt.colorbar()
        plt.figure()
        plt.title("Step 2: Post-bleach mask (intensity-based)")
        plt.imshow(mask_int, cmap='gray')
        plt.show()

    # Step 3: Calculate bleached region
    mask_roi2 = copy.deepcopy(mask_roi1)
    mask_roi2[mask_int == True] = False

    if log_level < 20:
        logger.debug(f"Step 3: Mask difference (potential bleached region)")
        plt.figure()
        plt.title("Step 3: Initial bleached region (ROI2)")
        plt.imshow(mask_roi2, cmap='gray')
        plt.show()

    # Step 4: Ensure single connected component
    label_image, label_num = measure.label(mask_roi2, return_num=True)
    if label_num > 1:        
        logger.debug(f"Step 4: Found {label_num} components, selecting largest")
        mask_roi2 = label_image == np.argmax(np.bincount(label_image.flat)[1:]) + 1
    elif label_num == 0:
        logger.debug(f"Step 4: No components found, reverting to ROI1")
        mask_roi2 = copy.deepcopy(mask_roi1)


    if log_level < 20:
        plt.figure()
        plt.title("Step 4: Final bleached region (ROI2)")
        plt.imshow(mask_roi2, cmap='gray')
        plt.show()

    return mask_roi1, mask_roi2


def refine_ROIs2(curr_img, ind_before, ind_after):
    """
    Alternative ROI refinement method (not yet implemented).
    
    Args:
        curr_img: 3D image array
        ind_before: Pre-bleach frame index
        ind_after: Post-bleach frame index
        
    Returns:
        tuple: (0, 0) placeholder return
    """
    logger.warning('The second method for ROI calculation is not implemented yet')
    return 0, 0

def get_background_(mic_img, area_cond):
    """
    Calculate background mask and size for FRAP analysis.
    
    Args:
        mic_img: 3D numpy array of microscopy images (time, height, width)
        area_cond: Area condition value for background size calculation
        
    Returns:
        tuple: (backg_mask, backg_size)
            backg_mask: Binary mask marking background pixels
            backg_size: Size of background region
    """
    avg_img = np.mean(mic_img, axis=0)
    thresh_mean = filters.threshold_mean(avg_img)
    backg_mask = avg_img < thresh_mean
    backg_size = int(np.sqrt(3*area_cond))
    return

def resample_contour(contour, num_points):
    if len(contour) < 3:
        return contour  # Can't interpolate a line or point
    x, y = contour[:, 1], contour[:, 0]
    tck, _ = splprep([x, y], s=0, per=True)
    u_new = np.linspace(0, 1, num_points)
    x_new, y_new = splev(u_new, tck)
    return np.stack((y_new, x_new), axis=-1)

def export_intensity(curr_img, mask_roi1, mask_roi2, file_path):
    """
    Calculate and export intensity measurements for FRAP analysis.
    
    Calculates mean intensities and areas for:
    - Total cell area
    - Bleached region
    - Unbleached region (total - bleached)
    
    Args:
        curr_img: 3D image array
        mask_roi1: Binary mask of total cell area
        mask_roi2: Binary mask of bleached region
        file_path: Path for exporting results
        
    Exports results to Excel file with measurements for each region.
    """
    # Calculate total area measurements
    img_area_1 = copy.deepcopy(curr_img)
    img_area_1[:, mask_roi1==False] = 0
    area_total_ = np.sum(mask_roi1)
    area_total = np.ones(len(curr_img))*area_total_
    mean_intensity_whole_area = np.sum(np.sum(img_area_1, axis=1), axis=1)/area_total_
    
    # Calculate bleached area measurements
    img_area_2 = copy.deepcopy(curr_img)
    img_area_2[:, mask_roi2==False] = 0
    area_bleached_ = np.sum(mask_roi2)
    area_bleached = np.ones(len(curr_img))*area_bleached_
    mean_intensity_bleached_area = np.sum(np.sum(img_area_2, axis=1), axis=1)/area_bleached_
    
    # Calculate unbleached area measurements
    mask_roi3 = copy.deepcopy(mask_roi1)
    mask_roi3[mask_roi2==True] = False
    img_area_3 = copy.deepcopy(curr_img)
    img_area_3[:, mask_roi3==False] = 0
    area_unbleached_ = np.sum(mask_roi3)
    area_unbleached = np.ones(len(curr_img))*area_unbleached_
    mean_intensity_unbleached_area = np.sum(np.sum(img_area_3, axis=1), axis=1)/area_unbleached_
    
    # Export results to Excel
    results = {
        'Area(all)': area_total,
        'Mean_intensity(all)': mean_intensity_whole_area,
        'Area(bleached)': area_bleached,
        'Mean_intensity(bleached)': mean_intensity_bleached_area,
        'Area(unbleached)': area_unbleached,
        'Mean_intensity(unbleached)': mean_intensity_unbleached_area
    }
    
    results = pd.DataFrame(data=results)
    excel_path = file_path[:-4]+'.xlsx'
    logger.info('wrote results to ' + excel_path)
    results.to_excel(excel_path)
    return

logger.info("Functions loaded successfully!")

[INFO] FRAP_Logger: Functions loaded successfully!


In [6]:
# -*- coding: utf-8 -*-
"""
ImageVisualizer class for interactive visualization and analysis of FRAP microscopy data.

This class provides functionality to:
- Load and display microscopy image stacks 
- Navigate through image slices using scroll wheel or slider
- Select regions of interest (ROIs) for FRAP analysis
- Apply different ROI detection methods
- Export intensity measurements
- Handle drift correction
- Support image supersampling for higher resolution

Key features:
- Interactive matplotlib-based GUI
- Multiple ROI selection methods
- Background correction
- Export to Excel
- Drift correction (optional)

Authors: damlatetiker, johannbrenner, jeremypflaum
Created: Nov 1, 2022
"""


class ImageVisualizer:
    """
    Class for interactive visualization and analysis of FRAP microscopy data.
    
    Attributes:
        _abort (bool): Flag to abort current processing
        file_path (str): Path to input microscopy file
        file (str): Input filename
        supersampling (int): Factor for image supersampling (default=2)
        drift_correction (bool): Enable drift correction (default=True)
        poly_full (PolygonSelector): Selector for full cell ROI
        poly_bleach (PolygonSelector): Selector for bleached region ROI
        result_exported (bool): Flag if results were exported
    """

    def __init__(self):
        self._abort = False
        return
        
    def __init__(self, path):
        """
        Initialize visualizer with input file path.
        
        Args:
            path (str): Path to microscopy image file
        """
        
        if log_level < 20:
            self.DEBUGGING = True
        else:
            self.DEBUGGING = False
        self.num_points_per_contour = NUM_POINTS_PER_CONTOUR  # Default value, can be updated via UI
        self._abort = False
        self.file_path = path   
        self.file = self.file_path.split('/')[-1]
        self.supersampling = 2
        self.drift_correction = True
        self.poly_full = None
        self.poly_bleach = None
        self.result_exported = False
        self.createFigure(self.file_path)
        self.update()
        self.connect()
        return

    def on_close_abort(self, event):
        """Handle window close event by aborting processing."""
        self._abort = True        
        self.fig.canvas.stop_event_loop()
        logger.info('Triggered abort')

    def createFigure(self, path):
        """
        Create matplotlib figure and initialize image display.
        
        Args:
            path (str): Path to image file to display
        """
        self.fig, self.ax = plt.subplots(figsize=(9,6))      
        self.mic_img = io.imread(path)

        self.slices, rows, cols = self.mic_img.shape

        self.mic_img_dim = (int(cols * self.supersampling), int(rows * self.supersampling))
        if self.supersampling > 1:
            logger.info('Supersampling image to a final resolution of {}x{}'.format(self.mic_img_dim[0], self.mic_img_dim[1]))
            self.mic_img_supersampled_tmp = np.zeros((self.slices,self.mic_img_dim[0], self.mic_img_dim[1]))
            for i in range(self.slices):
                self.mic_img_supersampled_tmp[i,:,:] = cv2.resize(self.mic_img[i,:,:],(self.mic_img_dim[0], self.mic_img_dim[1]), interpolation = cv2.INTER_LANCZOS4) 
            self.mic_img = self.mic_img_supersampled_tmp
            del self.mic_img_supersampled_tmp

        axslid = self.fig.add_axes([0.3, 0.03, 0.4, 0.03])
        self.slider = Slider(ax=axslid, label='Slice', valmin=0, valmax=self.mic_img.shape[0]-1, valinit=0, valstep=1, initcolor='none', color='lightgrey')
        

        # Add textbox for number of points per contour
        ax_textbox = self.fig.add_axes([0.85, 0.03, 0.1, 0.03])  # x, y, width, height
        self.textbox_points = TextBox(ax_textbox, 'Points', initial=str(self.num_points_per_contour))

        def submit_points(text):
            try:
                val = int(text)
                if val >= 4:
                    self.num_points_per_contour = val + 1
                    if hasattr(self, 'mask_roi1') and hasattr(self, 'mask_roi2'):
                        self.draw_ROI(self.mask_roi1, self.mask_roi2)
                else:
                    self.num_points_per_contour = 4 + 1
            except ValueError:
                pass

        self.textbox_points.on_submit(submit_points)
                
        self.ax.set_aspect('equal')
        self.ax.set_title('use scroll wheel to navigate images')
        self.selector = RectangleSelector(self.ax,self.select_roi)
        
        self.curr_img = self.mic_img
        self.ind = 0

        self.im = self.ax.imshow(self.mic_img[self.ind, :, :], cmap='gray')
        self.curr_img_bbox = [-0.5, self.mic_img_dim[1]+.5, self.mic_img_dim[0]+.5, -0.5]

        self.update()
        return
    
        
    def connect(self):
        """Connect matplotlib event handlers."""
        self.fig.canvas.mpl_connect('scroll_event', self.on_scroll)
        self.fig.canvas.mpl_connect('key_press_event', self.on_press)
        self.slider.on_changed(self.update_from_slider)
        self.on_close_abort_CID = self.fig.canvas.mpl_connect('close_event', self.on_close_abort)    
        return
    
    def connect2(self):
        """Connect button event handlers."""
        self.but1.on_clicked(self.ROI_method1)
        self.but2.on_clicked(self.ROI_method2)
        self.but3.on_clicked(self.export)
        self.but4.on_clicked(self.next_stack)
        return
    
    def next_stack(self,event):
        """Handle next stack button click."""
        if self.result_exported == False:
            self.ax.set_title("You didn't export your latest results!\nIf that was on purpose, click Next Stack again", color='red')  
            self.result_exported = True
            self.fig.canvas.draw_idle()
            return
        self._abort = False        
        self.fig.canvas.stop_event_loop()        
        self.fig.canvas.mpl_disconnect(self.on_close_abort_CID)    
        plt.close()
        logger.info('Triggered continue')
    
    def on_scroll(self, event):
        """Handle mouse scroll events for slice navigation."""
        if event.button == 'up':
            self.ind = (self.ind + 1) % self.slices
        else:
            self.ind = (self.ind - 1) % self.slices
        self.slider.set_val(self.ind)
        self.update()
        return
        
    def on_press(self, event):
        """
        Handle keyboard events.
        
        Supported keys:
        - up/right: Next slice
        - down/left: Previous slice  
        - shift+alt: Reset view
        - enter: Start ROI detection
        """
        if event.key == 'up' or event.key == 'right':
            self.ind = (self.ind + 1) % self.slices
        elif event.key == 'down'or event.key == 'left':
            self.ind = (self.ind - 1) % self.slices
        elif event.key == 'shift+alt':
            self.curr_img = self.mic_img
            self.curr_img_bbox = [-0.5, self.mic_img_dim[1] + .5, self.mic_img_dim[0] + .5, -0.5]
            self.ax.set_title('use scroll wheel to navigate images')
            for line in self.ax.get_lines():
                line.remove()
            self.selector.set_active(True)
        elif event.key == 'enter':
            self.ind_before, self.ind_after = detect_frap(self.curr_img)    
            self.ax.set_title('Now showing the ROIs calculated with method 1.\n' + f + '\n')
            
            self.selector.set_active(False)
            ax_met1 = self.fig.add_axes([0.8, 0.8, 0.1, 0.06])
            ax_met2 = self.fig.add_axes([0.8, 0.7, 0.1, 0.06])
            ax_met3 = self.fig.add_axes([0.8, 0.6, 0.1, 0.06])            
            ax_met4 = self.fig.add_axes([0.8, 0.5, 0.1, 0.06])
            self.but1 = Button(ax_met1, 'ROI method 1')
            self.but2 = Button(ax_met2, 'ROI method 2 \n not implem. ')
            self.but3 = Button(ax_met3, 'Export')
            self.but4 = Button(ax_met4, 'Next Stack')

            self.mask_roi1, self.mask_roi2 = refine_ROIs(self.curr_img, self.ind_before, self.ind_after)
            self.area_cond = np.sum(self.mask_roi1)
            self.get_background()
            self.draw_ROI(self.mask_roi1, self.mask_roi2)
            self.connect2()
        self.slider.set_val(self.ind)
        self.update()
        return
  
    def ROI_method1(self, event):
        """Apply ROI detection method 1."""
        self.ax.set_title('Now showing the ROIs calculated with method 1.\nTo swith press button ROI method 2.\nTo see the background ROI press shift alt (not implemented yet)\nTo export press button Export')
        self.mask_roi1, self.mask_roi2 = refine_ROIs(self.curr_img, self.ind_before, self.ind_after)
        self.draw_ROI(self.mask_roi1, self.mask_roi2)
        return
    
    def ROI_method2(self, event):
        """Apply ROI detection method 2."""
        self.ax.set_title('Now showing the ROIs calculated with method 2.\nTo swith press button ROI method 1.\nTo see the background ROI press shift alt (not implemented yet)\nTo export press button Export')
        self.mask_roi1, self.mask_roi2 = refine_ROIs2(self.curr_img, self.ind_before, self.ind_after)
        self.draw_ROI(self.mask_roi1, self.mask_roi2)
        return
    
    def get_background(self):
        """Calculate image background."""
        get_background_(self.mic_img, self.area_cond)
        return
    
    def verts_to_mask(self, verts, shape):
        """Convert polygon vertices to binary mask."""
        mask = np.zeros(shape, dtype=np.uint8)
        vertices_normalized = np.array(verts)
        vertices_normalized[:, 0] -= self.curr_img_bbox[0] + 0.5
        vertices_normalized[:, 1] -= self.curr_img_bbox[3] + 0.5
        cv2.fillPoly(mask, [np.int32(vertices_normalized)], 1)
        return mask > 0
        
    def export(self, event):
        """Export intensity measurements to file."""
        if self.poly_full and self.poly_bleach:
            mask1 = self.verts_to_mask(self.poly_full.verts, self.curr_img.shape[1:])
            mask2 = self.verts_to_mask(self.poly_bleach.verts, self.curr_img.shape[1:])
            
            if self.DEBUGGING:
                fig, axs = plt.subplots(1, 2, figsize=(8, 4))
                axs[0].imshow(mask1, cmap='gray')
                axs[0].set_title("Export Mask 1 (Full ROI)")
                axs[1].imshow(mask2, cmap='gray')
                axs[1].set_title("Export Mask 2 (Bleach ROI)")
                plt.tight_layout()
                plt.show()
                
            export_intensity(self.curr_img, mask1, mask2, self.file_path)
            self.result_exported = True
        else:
            self.ax.set_title("No ROI selected for export", color='red')
        return
        
    def draw_ROI(self, mask1, mask2):
        """
        Draw ROI contours on image.
    
        Args:
            mask1: Mask for full cell ROI
            mask2: Mask for bleached region ROI
        """
        for line in self.ax.lines[:]:
            line.remove()
        if self.poly_full:
            self.poly_full.set_active(False)
            self.poly_full.disconnect_events()
            self.poly_full = None

        if self.poly_bleach:
            self.poly_bleach.set_active(False)
            self.poly_bleach.disconnect_events()
            self.poly_bleach = None
        
        logger.debug(f"mask2 dtype: {mask2.dtype}, unique values: {np.unique(mask2)}")
        if self.DEBUGGING:
            # Debugging output for mask2
            fig, ax = plt.subplots()
            ax.set_title("Debug: mask2 before contour extraction")
            ax.imshow(mask2, cmap='gray')
            plt.show()
    
        contours1 = measure.find_contours(mask1 == 1)
        contours2 = measure.find_contours(mask2 == 1)
        
        if self.DEBUGGING:            
            # Save contour2 to a file            
            save_path = "./debug_mask2.npy"
            np.save(save_path, mask2)
            logger.debug(f"Saved Mask2 to {save_path}")
        
        if not contours1 or not contours2:
            raise ValueError("No contours found in one or both masks.")
        
        contour1 = max(contours1, key=len)
        contour2 = max(contours2, key=len)
        bbox = self.curr_img_bbox
        
        logger.debug(f"Bounding box: {bbox}")
        logger.debug(f"Contour1 shape: {contour1.shape}")
        logger.debug(f"Contour2 shape: {contour2.shape}")
    
        if self.DEBUGGING:    
            fig, ax = plt.subplots()
            ax.set_title("Contours from Masks")
            ax.imshow(mask1, cmap='gray', alpha=0.5)
            ax.plot(contour1[:, 1], contour1[:, 0], 'r--', label='ROI1 (full)')
            ax.plot(contour2[:, 1], contour2[:, 0], 'b--', label='ROI2 (bleach)')
            ax.legend()
            plt.show()
    
        poly_full_props = dict(color='r', linestyle='--', linewidth=2, alpha=0.5)
        self.poly_full = PolygonSelector(self.ax, onselect=self.on_select_poly1, props=poly_full_props, grab_range=10)
    
        #poly_full_vertices = [
        #    (contour1[i, 1] + bbox[0] + 0.5, contour1[i, 0] + bbox[3] + 0.5)
        #    for i in range(1, len(contour1[:, 1]), 4)
        #]
        contour1_resampled = resample_contour(contour1, self.num_points_per_contour)
        poly_full_vertices = [
            (x + bbox[0] + 0.5, y + bbox[3] + 0.5) for y, x in contour1_resampled
        ]
        self.poly_full.verts = poly_full_vertices
    
        poly_bleach_props = dict(color='b', linestyle='--', linewidth=2, alpha=0.5)
        self.poly_bleach = PolygonSelector(self.ax, onselect=self.on_select_poly2, props=poly_bleach_props, grab_range=10)
    
        # poly_bleach_vertices = [
        #     (contour2[i, 1] + bbox[0] + 0.5, contour2[i, 0] + bbox[3] + 0.5)
        #     for i in range(1, len(contour2[:, 1]), 4)
        # ]
        contour2_resampled = resample_contour(contour2, self.num_points_per_contour)
        poly_bleach_vertices = [
            (x + bbox[0] + 0.5, y + bbox[3] + 0.5) for y, x in contour2_resampled
        ]
        self.poly_bleach.verts = poly_bleach_vertices
    
        logger.debug(f"PolygonSelector full ROI vertices: {len(poly_full_vertices)} points")
        logger.debug(f"PolygonSelector bleach ROI vertices: {len(poly_bleach_vertices)} points")
    
        self.im.set_extent(bbox)
        return
        
    def update(self):
        """Update image display."""
        self.im.set_extent(self.curr_img_bbox)
        self.im.set_data(self.curr_img[self.ind, :, :])
        self.ax.set_aspect('equal')
        self.ax.set_ylabel('slice %s' % self.ind)
        self.im.axes.figure.canvas.draw()
        self.fig.canvas.draw_idle()
        return

    def update_from_slider(self, val):
        """Update display based on slider value."""
        self.ind = val
        self.update()
        return
    
    def select_roi(self, eclick, erelease):
        """Handle ROI selection via rectangle."""
        self.curr_img = self.mic_img[:, int(eclick.ydata):int(erelease.ydata), int(eclick.xdata):int(erelease.xdata)]
        self.curr_img_bbox = [int(eclick.xdata)-0.5, int(erelease.xdata)-0.5, int(erelease.ydata)-0.5, int(eclick.ydata)-0.5]
        self.ax.set_title('Are you happy with the current ROI?\nTo reset press shift and alt\nTo continue with automated selection press enter')
        self.update()
        return
    
    def on_select_poly1(self, vertices):
        """Handle selection of full cell ROI polygon."""
        self.redraw_roi1(vertices)
        self.fig.canvas.draw_idle() 
        
    def on_select_poly2(self, vertices):
        """Handle selection of bleached region ROI polygon."""
        self.redraw_roi2(vertices)
        self.fig.canvas.draw_idle()
    
    def redraw_roi1(self, vertices):
        """Redraw full cell ROI mask from vertices."""
        black_image = np.zeros((self.mask_roi1.shape[0],self.mask_roi1.shape[1]),dtype=np.uint8)
        vertices_normalized = np.array([*vertices])
        vertices_normalized[:,0] = vertices_normalized[:,0] - self.curr_img_bbox[0] -0.5
        vertices_normalized[:,1] = vertices_normalized[:,1] - self.curr_img_bbox[3] -0.5
        self.mask_roi1= cv2.fillPoly(black_image, pts =np.int32([vertices_normalized]), color=(255,255,255))
        self.mask_roi1 = self.mask_roi1 > 128

    def redraw_roi2(self, vertices):
        """Redraw bleached region ROI mask from vertices."""
        black_image = np.zeros((self.mask_roi2.shape[0],self.mask_roi2.shape[1]),dtype=np.uint8)
        vertices_normalized = np.array([*vertices])
        vertices_normalized[:,0] = vertices_normalized[:,0] - self.curr_img_bbox[0] -0.5
        vertices_normalized[:,1] = vertices_normalized[:,1] - self.curr_img_bbox[3] -0.5
        self.mask_roi2 = cv2.fillPoly(black_image, pts =np.int32([vertices_normalized]), color=(255,255,255))
        self.mask_roi2 = self.mask_roi2 > 128

    def get_canvas(self):
        """Return the matplotlib canvas."""
        return self.fig.canvas

In [7]:
def checkFileListOverwrite(file_dir):
    """
    Get list of TIFF files in directory, excluding hidden files.
    
    Args:
        file_dir (str): Directory path to check for files
        
    Returns:
        list: List of TIFF filenames found in directory
    """
    # Get all .tif files, excluding hidden files that start with ._
    file_list = [f for f in listdir(file_dir) 
                 if isfile(join(file_dir, f)) 
                 and f.endswith('.tif') 
                 and not f.startswith('._')]
    
    return file_list

# Set directory path and get list of TIFF files
file_dir = './FloData'
file_list = checkFileListOverwrite(file_dir)
pprint(file_list)


['FRapWT500nMN1_01_D3D.tif', 'FRapWT500nMN1_02_D3D.tif']


In [8]:
# Iterate through each TIFF file in the directory
for f in file_list:
    # Construct full file path by joining directory and filename
    file_path = file_dir + '/' + f
    logger.info(f"Processing file: {file_path}")
    
    try:
        # Create ImageVisualizer instance for the current TIFF file
        # This will display the image and allow ROI selection
        vis = ImageVisualizer(file_path)
        
        # Display the visualization window and block until closed
        # This ensures user can interact with the ROI selection
        plt.show(block=True)
        
        # Note: Threading code below is commented out but kept for reference
        # Could be used to run visualization in separate thread if needed
        #t1 = Thread(target = vis.get_canvas().start_event_loop(), name='ImageVisualizer')
        #t1.daemon = True 
        #t1.start()
        
    except Exception as e:
        # Log any errors that occur during visualization
        # This helps with debugging if image loading or display fails
        logging.info('Error occurred during execution of ImageVisualizer:')
        logging.info(f'Exception details: {str(e)}')
        sys.exit()  # Exit program on error

[INFO] FRAP_Logger: Processing file: ./FloData/FRapWT500nMN1_01_D3D.tif
[INFO] FRAP_Logger: Supersampling image to a final resolution of 480x480
[DEBUG] FRAP_Logger: Index before bleach: 0
[DEBUG] FRAP_Logger: Index after bleach: 1
[DEBUG] FRAP_Logger: Step 1: Pre-bleach average threshold = 21129.232
[DEBUG] FRAP_Logger: Step 2: Post-bleach threshold = 16590.570
[DEBUG] FRAP_Logger: Step 3: Mask difference (potential bleached region)
[DEBUG] FRAP_Logger: Step 4: Found 3 components, selecting largest
[DEBUG] FRAP_Logger: mask2 dtype: bool, unique values: [False  True]
[DEBUG] FRAP_Logger: Saved Mask2 to ./debug_mask2.npy
[DEBUG] FRAP_Logger: Bounding box: [173.5, 315.5, 297.5, 158.5]
[DEBUG] FRAP_Logger: Contour1 shape: (219, 2)
[DEBUG] FRAP_Logger: Contour2 shape: (145, 2)
[DEBUG] FRAP_Logger: PolygonSelector full ROI vertices: 15 points
[DEBUG] FRAP_Logger: PolygonSelector bleach ROI vertices: 15 points
[INFO] FRAP_Logger: wrote results to ./FloData/FRapWT500nMN1_01_D3D.xlsx
[INFO] FR